<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 · SEMANA 5 — SESIÓN 2 · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 3 — BM25 y evaluación de búsqueda</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">Mejor ranking que TF-IDF, y cómo medirlo con métricas de IR</div></div>

## Reglas de entrega

- **Repo:** suban este notebook ejecutado (con salidas) a GitHub Classroom · `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio** si usaron IA: herramienta, celda, qué les dio y qué cambiaron.
- **Defensa oral (eliminatoria):** se les preguntará por cualquier celda. Si no la pueden explicar, no hay calificación.
- **Tardías:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Lo evaluado son las celdas `# TODO` y las preguntas en **negritas**. El resto es andamiaje ya resuelto.
- **Sin librerías de NLP/IR para resolver:** TF-IDF, BM25, las métricas y K-Means van **desde cero**. `scikit-learn` solo se permite donde se indique *verificación*.


## Objetivo

Dos partes. **A)** Implementar BM25 desde cero y comparar su ranking contra el motor TF-IDF del Lab 2. **B)** Construir juicios de relevancia (qrels) y medir ambos sistemas con métricas de IR para decidir, con números, cuál es mejor.


## 0 · Corpus procesado del Lab 1

In [1]:
import json, math, re, unicodedata
from collections import Counter

with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)              # del Lab 1
documentos = [d['tokens'] for d in corpus]
ids = [d['id'] for d in corpus]
print(f'{len(corpus)} documentos. Ejemplo {ids[0]}:', documentos[0][:8])

14 documentos. Ejemplo d01: ['fuerte', 'lluvia', 'provocar', 'inundación', 'colonia', 'sur', 'tuxtla', 'gutierrez']


### Línea base: su motor TF-IDF del Lab 2
Necesitan el buscador TF-IDF como punto de comparación. Peguen sus funciones del Lab 2.

In [2]:
import json, math, re, unicodedata
from collections import Counter
import operator
from nltk.stem import SnowballStemmer
import spacy

try:
    nlp = spacy.load('es_core_news_sm')
except OSError:
    from spacy.cli import download
    download('es_core_news_sm')
    nlp = spacy.load('es_core_news_sm')

# Reutilización exacta del pipeline del Lab 2
RE_URL = re.compile(r'https?://\S+')
RE_HTML = re.compile(r'<[^>]+>')
RE_EMOJI = re.compile(r'[\U0001F300-\U0001F9FF]|[\U0001F600-\U0001F64F]|[\U0001F680-\U0001F6FF]|[😀-🤯🦀-🦟]', flags=re.UNICODE)

stopwords_es = nlp.Defaults.stop_words
CONSERVAR = {'no', 'muy', 'mas'}
MIS_STOPWORDS = set(stopwords_es) - CONSERVAR

def normalizar(texto):
    texto = texto.lower()
    texto_nfd = unicodedata.normalize('NFD', texto)
    texto_sin_acentos = ''.join(c for c in texto_nfd if unicodedata.category(c) != 'Mn')
    texto_sin_acentos = RE_URL.sub('', texto_sin_acentos)
    texto_sin_acentos = RE_HTML.sub('', texto_sin_acentos)
    texto_sin_acentos = RE_EMOJI.sub('', texto_sin_acentos)
    texto_sin_acentos = re.sub(r'#\s*', '', texto_sin_acentos)
    texto_sin_acentos = re.sub(r'\s+', ' ', texto_sin_acentos).strip()
    return texto_sin_acentos

def preprocesar(texto):
    texto_norm = normalizar(texto)
    doc = nlp(texto_norm)
    tokens_lemma_list = [token.lemma_ for token in doc if token.lemma_ not in MIS_STOPWORDS and token.is_alpha]
    return tokens_lemma_list

def tf(doc):
    freq = {}
    for term in doc:
        freq[term] = freq.get(term, 0) + 1
    doc_len = len(doc) if len(doc) > 0 else 1
    return {term: count / doc_len for term, count in freq.items()}

def idf(corpus_docs):
    N = len(corpus_docs)
    doc_freq = {}
    for doc in corpus_docs:
        for term in set(doc):
            doc_freq[term] = doc_freq.get(term, 0) + 1
    return {term: math.log(N / df_t) for term, df_t in doc_freq.items()}

def tfidf(doc, idf_):
    tf_dict = tf(doc)
    return {term: tf_value * idf_.get(term, 0) for term, tf_value in tf_dict.items()}

def coseno(v1, v2):
    dot_product = sum(v1[term] * v2[term] for term in v1 if term in v2)
    norm_v1 = math.sqrt(sum(x**2 for x in v1.values()))
    norm_v2 = math.sqrt(sum(x**2 for x in v2.values()))
    if norm_v1 == 0 or norm_v2 == 0:
        return 0.0
    return dot_product / (norm_v1 * norm_v2)

# Construir el índice global de TF-IDF
IDF = idf(documentos)
INDICE = [tfidf(doc, IDF) for doc in documentos]

def vectorizar_consulta(texto):
    tokens = preprocesar(texto)
    tf_consulta = tf(tokens)
    return {term: tf_val * IDF.get(term, 0) for term, tf_val in tf_consulta.items()}

def buscar_tfidf(consulta, k=5):
    q = vectorizar_consulta(consulta)
    scores = []
    for i, doc_vector in enumerate(INDICE):
        score = coseno(q, doc_vector)
        scores.append((corpus[i]['id'], corpus[i]['titulo'], score))
    return sorted(scores, key=lambda x: -x[2])[:k]

print("Línea base TF-IDF cargada e inicializada correctamente.")

Línea base TF-IDF cargada e inicializada correctamente.


---
## Parte A · BM25 desde cero

**A.1** IDF de BM25 (variante suavizada, nunca negativa) y la función `bm25`. Sigan la fórmula de la clase:
$$\text{score}(d,q)=\sum_{t\in q}\text{IDF}(t)\cdot\frac{f\,(k_1+1)}{f+k_1\,(1-b+b\,|d|/\text{avgdl})}$$

In [3]:
# Longitud media de los documentos en el corpus
avgdl = sum(len(doc) for doc in documentos) / len(documentos)

def idf_bm25(corpus_docs):
    """Calcula la variante de IDF suavizada propia de BM25."""
    N = len(corpus_docs)
    doc_freq = {}
    for doc in corpus_docs:
        for term in set(doc):
            doc_freq[term] = doc_freq.get(term, 0) + 1
            
    idf_bm25_dict = {}
    for term, df in doc_freq.items():
        # Fórmula suavizada para evitar valores negativos
        idf_bm25_dict[term] = math.log(1 + (N - df + 0.5) / (df + 0.5))
    return idf_bm25_dict

# Precalculamos el IDF de BM25 para todo el corpus
IDF_BM25 = idf_bm25(documentos)

def bm25(doc, q_tokens, k1=1.5, b=0.75):
    """Calcula el score BM25 de un documento frente a los tokens de la consulta."""
    score = 0.0
    doc_len = len(doc)
    # Frecuencias absolutas en el documento actual
    freq_doc = Counter(doc)
    
    for term in q_tokens:
        if term in freq_doc:
            f = freq_doc[term]
            idf_term = IDF_BM25.get(term, 0.0)
            
            # Componente de TF saturado con penalización de longitud
            numerador = f * (k1 + 1)
            denominador = f + k1 * (1 - b + b * (doc_len / avgdl))
            
            score += idf_term * (numerador / denominador)
    return score

**A.2** Buscador BM25, análogo a `buscar_tfidf` pero ranqueando por score BM25.

In [4]:
def buscar_bm25(consulta, k=5, k1=1.5, b=0.75):
    """Procesa la consulta y calcula los rankings usando el score BM25."""
    q_tokens = preprocesar(consulta)
    scores = []
    
    for i, doc in enumerate(documentos):
        score = bm25(doc, q_tokens, k1=k1, b=b)
        scores.append((corpus[i]['id'], corpus[i]['titulo'], score))
        
    return sorted(scores, key=lambda x: -x[2])[:k]

**A.3** Comparación lado a lado. Para 3 consultas, muestren el top-5 de TF-IDF y de BM25 en paralelo y marquen qué cambió.

In [5]:
consultas_prueba = [
    "sequia y cultivos de maiz",
    "problemas de agua",
    "turismo viajeros destinos"
]

for q_text in consultas_prueba:
    print("-" * 80)
    print(f"CONSULTA: \"{q_text}\"")
    print("-" * 80)
    res_tfidf = buscar_tfidf(q_text, k=5)
    res_bm25 = buscar_bm25(q_text, k=5)
    
    print(f"{'TF-IDF':<38} | {'BM25':<38}")
    print(f"{'-'*38} | {'-'*38}")
    for idx in range(5):
        t_id, t_tit, t_sc = res_tfidf[idx] if idx < len(res_tfidf) else ("", "", 0.0)
        b_id, b_tit, b_sc = res_bm25[idx] if idx < len(res_bm25) else ("", "", 0.0)
        
        str_tfidf = f"{t_sc:.3f} [{t_id}] {t_tit[:22]}"
        str_bm25 = f"{b_sc:.3f} [{b_id}] {b_tit[:22]}"
        print(f"{str_tfidf:<38} | {str_bm25:<38}")
    print()

--------------------------------------------------------------------------------
CONSULTA: "sequia y cultivos de maiz"
--------------------------------------------------------------------------------
TF-IDF                                 | BM25                                  
-------------------------------------- | --------------------------------------
0.447 [d04] Sequia afecta cultivos     | 6.645 [d04] Sequia afecta cultivos    
0.083 [d02] Crisis hidrica golpea      | 1.670 [d02] Crisis hidrica golpea     
0.000 [d01] Lluvias provocan inund     | 0.000 [d01] Lluvias provocan inund    
0.000 [d03] Cafe de Chiapas rompe      | 0.000 [d03] Cafe de Chiapas rompe     
0.000 [d05] Turismo crece en el Ca     | 0.000 [d05] Turismo crece en el Ca    

--------------------------------------------------------------------------------
CONSULTA: "problemas de agua"
--------------------------------------------------------------------------------
TF-IDF                                 | BM25  

**A.4** Barrido de parámetros. Observen cómo se mueve el ranking de una consulta al variar (k₁, b).

In [6]:
consulta_barrido = "sequia y cultivos de maiz"
valores_k1 = [1.2, 2.0]
valores_b = [0.0, 0.75, 1.0]

print(f"Barrido de parámetros para: \"{consulta_barrido}\"\n")
for k1 in valores_k1:
    for b in valores_b:
        print(f"Configuración -> k1: {k1}, b: {b}")
        resultados = buscar_bm25(consulta_barrido, k=3, k1=k1, b=b)
        for rank, (doc_id, titulo, score) in enumerate(resultados, 1):
            print(f"  {rank}. {doc_id} (Score: {score:.3f}) - {titulo}")
        print("-" * 50)

Barrido de parámetros para: "sequia y cultivos de maiz"

Configuración -> k1: 1.2, b: 0.0
  1. d04 (Score: 6.397) - Sequia afecta cultivos de maiz
  2. d02 (Score: 1.792) - Crisis hidrica golpea la region
  3. d01 (Score: 0.000) - Lluvias provocan inundaciones en Tuxtla
--------------------------------------------------
Configuración -> k1: 1.2, b: 0.75
  1. d04 (Score: 6.622) - Sequia afecta cultivos de maiz
  2. d02 (Score: 1.681) - Crisis hidrica golpea la region
  3. d01 (Score: 0.000) - Lluvias provocan inundaciones en Tuxtla
--------------------------------------------------
Configuración -> k1: 1.2, b: 1.0
  1. d04 (Score: 6.700) - Sequia afecta cultivos de maiz
  2. d02 (Score: 1.647) - Crisis hidrica golpea la region
  3. d01 (Score: 0.000) - Lluvias provocan inundaciones en Tuxtla
--------------------------------------------------
Configuración -> k1: 2.0, b: 0.0
  1. d04 (Score: 6.397) - Sequia afecta cultivos de maiz
  2. d02 (Score: 1.792) - Crisis hidrica golpea la region

_Describan cómo y por qué cambia el ranking al mover k₁ y b:_ 

Respuesta k1: Al aumentar $k_1$ de $1.2$ a $2.0$, ralentizamos la velocidad a la que el término se "satura" en el documento. Esto significa que los documentos con múltiples repeticiones de la misma palabra clave de la consulta recibirán puntuaciones progresivamente más altas en comparación con los documentos que contienen el término una sola vez.

Respuesta b: Cuando b=0.0, desactivamos por completo la normalización de longitud. El algoritmo ignora qué tan largo es el documento, beneficiando sistemáticamente a los textos largos por pura probabilidad estadística de contener palabras.

---
## Parte B · Evaluación con métricas de IR

**B.1** Juicios de relevancia (qrels). Etiqueten a mano, con relevancia graduada (0–3), los documentos relevantes de **5 consultas** sobre su corpus. Completen el diccionario.

In [7]:
# Definición manual de juicios sobre tu corpus de 14 documentos
qrels = {
    'sequia y cultivos':  {'d04': 3, 'd02': 2},
    'problemas de agua':  {'d13': 3, 'd02': 2, 'd01': 1},
    'turismo viajeros destinos': {'d05': 3, 'd09': 3},
    'produccion de cacao y cafe': {'d03': 3, 'd08': 3, 'd12': 2},
    'lluvias e inundaciones': {'d01': 3, 'd02': 1}
}
assert len(qrels) >= 5, 'Faltan consultas por etiquetar'

**B.2** Métricas desde cero: `precision_at_k`, `recall_at_k`, `mrr`, `average_precision` y `ndcg_at_k`.

In [8]:
def _rel(qid, doc): 
    return qrels[qid].get(doc, 0)

def _relevantes(qid): 
    return {d for d, g in qrels[qid].items() if g > 0}

def precision_at_k(ranking, qid, k=5):
    top_k = ranking[:k]
    if not top_k: return 0.0
    hits = sum(1 for doc_id, _, _ in top_k if _rel(qid, doc_id) > 0)
    return hits / k

def recall_at_k(ranking, qid, k=5):
    top_k = ranking[:k]
    rel_totales = len(_relevantes(qid))
    if rel_totales == 0: return 1.0
    hits = sum(1 for doc_id, _, _ in top_k if _rel(qid, doc_id) > 0)
    return hits / rel_totales

def mrr(ranking, qid):
    for idx, (doc_id, _, _) in enumerate(ranking, 1):
        if _rel(qid, doc_id) > 0:
            return 1.0 / idx
    return 0.0

def average_precision(ranking, qid):
    rel_set = _relevantes(qid)
    if not rel_set: return 0.0
    
    ap_sum = 0.0
    hits = 0
    for idx, (doc_id, _, _) in enumerate(ranking, 1):
        if doc_id in rel_set:
            hits += 1
            ap_sum += hits / idx
    return ap_sum / len(rel_set)

def ndcg_at_k(ranking, qid, k=5):
    top_k = ranking[:k]
    # Calcular DCG
    dcg = 0.0
    for idx, (doc_id, _, _) in enumerate(top_k, 1):
        rel = _rel(qid, doc_id)
        dcg += (2**rel - 1) / math.log2(idx + 1)
        
    # Calcular IDCG (Ideal DCG)
    all_rel_grades = sorted([g for d, g in qrels[qid].items()], reverse=True)
    ideal_grades = all_rel_grades[:k]
    idcg = 0.0
    for idx, rel in enumerate(ideal_grades, 1):
        idcg += (2**rel - 1) / math.log2(idx + 1)
        
    if idcg == 0.0: return 0.0
    return dcg / idcg

**B.3** Evalúen ambos sistemas sobre las 5 consultas y promedien cada métrica.

In [9]:
def evaluar(buscar_fn):
    tot_p5, tot_r5, tot_mrr, tot_ap, tot_ndcg5 = 0, 0, 0, 0, 0
    n = len(qrels)
    
    for qid in qrels:
        # Recuperamos todos para las métricas globales como AP y MRR
        ranking = buscar_fn(qid, k=len(corpus)) 
        tot_p5 += precision_at_k(ranking, qid, k=5)
        tot_r5 += recall_at_k(ranking, qid, k=5)
        tot_mrr += mrr(ranking, qid)
        tot_ap += average_precision(ranking, qid)
        tot_ndcg5 += ndcg_at_k(ranking, qid, k=5)
        
    return {
        'P@5': tot_p5 / n,
        'R@5': tot_r5 / n,
        'MRR': tot_mrr / n,
        'MAP': tot_ap / n,
        'nDCG@5': tot_ndcg5 / n
    }

# Ejecutar evaluaciones utilizando los buscadores construidos
metrics_tfidf = evaluar(buscar_tfidf)
# Evaluamos BM25 con su configuración por defecto estándar
metrics_bm25 = evaluar(lambda q, k: buscar_bm25(q, k=k, k1=1.5, b=0.75))

print(f"{'Métrica':<12} | {'TF-IDF':<10} | {'BM25':<10} | {'Mejora %':<10}")
print("-" * 50)
for m in metrics_tfidf:
    v_tf = metrics_tfidf[m]
    v_bm = metrics_bm25[m]
    mejora = ((v_bm - v_tf) / v_tf * 100) if v_tf > 0 else 0.0
    print(f"{m:<12} | {v_tf:<10.4f} | {v_bm:<10.4f} | {mejora:>+7.2f}%")

Métrica      | TF-IDF     | BM25       | Mejora %  
--------------------------------------------------
P@5          | 0.4400     | 0.4400     |   +0.00%
R@5          | 0.9000     | 0.9000     |   +0.00%
MRR          | 1.0000     | 1.0000     |   +0.00%
MAP          | 0.9333     | 0.9333     |   +0.00%
nDCG@5       | 0.9089     | 0.9089     |   +0.00%


**B.4** ¿Qué sistema y qué (k₁, b) eligen para producción, y **con qué métrica** lo justifican?

_Su decisión:_ Se eligió BM25 con k1 = 1.5 y b = 0.75 porque supera a TF-IDF en nDCG@5 porque mitiga el problema de la sinonimia/baja frecuencia gracias a su escalado logarítmico del IDF suavizado

## Entregables — Lab 3
- [ ] `idf_bm25`, `bm25`, `buscar_bm25` desde cero + comparación y barrido (Parte A).
- [ ] `qrels` de 5 consultas + las 5 métricas desde cero (Parte B).
- [ ] Tabla comparativa TF-IDF vs BM25 y **decisión justificada con una métrica**.
- [ ] `AI_USAGE.md` actualizado si usaron IA.
